# Pennylane Fundamentals

## Gradients and Optimization

In [1]:
import pennylane as qml
from pennylane import numpy as np

### Circuits as functions

In [2]:
dev = qml.device("default.qubit", wires = 3)

@qml.qnode(dev)
def circuit_as_function(params):
    """
    Implements the circuit shown in the codercise statement.
    Args:
    - params (np.ndarray): [theta_0, theta_1, theta_2, theta_3]
    Returns:
    - (np.tensor): <Z0>
    """

    ####################
    ###YOUR CODE HERE###
    ####################
    qml.RX(params[0], wires = 0)
    qml.CNOT(wires = [0,1])
    qml.CNOT(wires = [1,2])
    qml.CNOT(wires = [2,0])
    qml.RY(params[1], wires = 0)
    qml.RY(params[2], wires = 1)
    qml.RY(params[3], wires = 2)

    return qml.expval(op = qml.Z(0))

angles = np.linspace(0, 4 * np.pi, 200)
output_values = np.array([circuit_as_function([0.5, t, 0.5, 0.5]) for t in angles])

### A strong entangler

In [3]:
dev = qml.device("default.qubit", wires = 4)

@qml.qnode(dev)
def strong_entangler(weights):
    """
    Applies Strongly Entangling Layers to the default initial state
    Args:
    - weights (np.ndarray): The weights argument for qml.StronglyEntanglingLayers
    Returns:
    - (np.tensor): <Z0>
    """
    qml.StronglyEntanglingLayers(weights=weights, wires=range(4))

    
    return qml.expval(qml.PauliZ(0))

shape = qml.StronglyEntanglingLayers.shape(n_layers=1, n_wires=4)
rng = np.random.default_rng(12345)
test_weights = rng.random(size=shape)


print("The output of your circuit with these weights is: ", strong_entangler(test_weights))


The output of your circuit with these weights is:  0.8805676541576933


### How to train your QNode

In [4]:
dev = qml.device("default.qubit", wires = 3)

@qml.qnode(dev)
def embedding_and_circuit(features, params):
    """
    A QNode that depends on trainable and non-trainable parameters
    Args:
    - features (np.ndarray): Non-trainable parameters in the AngleEmbedding routine
    - params (np.ndarray): Trainable parameters for the rest of the circuit
    Returns:
    - (np.tensor): <Z0>
    """

    ####################
    ###YOUR CODE HERE###
    ####################
    qml.AngleEmbedding(features = features, wires = range(3))
    qml.CNOT(wires = [0,1])
    qml.CNOT(wires = [1,2])
    qml.CNOT(wires = [2,0])
    qml.AngleEmbedding(features = params, wires = range(3), rotation = 'Y')


    return qml.expval(qml.PauliZ(0))

features = np.array([0.3,0.4,0.6], requires_grad = False)
params = np.array([0.4,0.7,0.9], requires_grad = True)
print("The gradient of the circuit is:", qml.jacobian(embedding_and_circuit)(features, params))


The gradient of the circuit is: [-2.96029765e-01  2.77555756e-17  0.00000000e+00]


### Differentiate and repeat

In [5]:
dev = qml.device("default.qubit", wires = 2)

@qml.qnode(dev, diff_method = "parameter-shift", max_diff = 2)
def circuit_for_hessian(params):
    """
    Implements the circuit shown in the codercise statement
    Args:
    - params (np.ndarray): [theta_0, theta_1, theta_2, theta_3]
    Returns:
    - np.tensor: <Z0xZ1>
    """

    ####################
    ###YOUR CODE HERE###
    ####################

    qml.RY(params[0], wires = 0)
    qml.IsingXX(params[1], wires = [0,1])
    qml.RX(params[2], wires = 0)
    qml.RX(params[3], wires = 1)

    return qml.expval(qml.Z(0) @ qml.Z(1))
    
test_params = np.array([0.1,0.2,0.3,0.4], requires_grad = True)
# Don't change test_params! 

hessian = qml.jacobian(qml.jacobian(circuit_for_hessian))(test_params)# Compute the Hessian
print("The hessian of the circuit is: \n", hessian)


The hessian of the circuit is: 
 [[-8.75527226e-01 -1.73472348e-17  2.71738709e-02  3.71405819e-02]
 [-1.38777878e-17 -2.22044605e-16  2.77555756e-17  5.55111512e-17]
 [ 2.71738709e-02  4.16333634e-17 -8.75527226e-01  1.14506063e-01]
 [ 3.71405819e-02  5.55111512e-17  1.14506063e-01 -8.75527226e-01]]


### At what cost ? 

In [6]:
def cost_function(params):
    """
    Computes the cost function given in the codercise, as a function of the
    parameters of circuit_as_function.
    Args:
    - params (np.ndarray): The parameters we pass to circuit_as_function
    Returns:
    - np.float: The cost function evaluated in params.
    """
    ################
    #YOUR CODE HERE#
    ################
    x = circuit_as_function(params)

    a, b, c, d = 1, -1/2, 1, 0

    cost = a*x**3 + b*x**2 + c*x + d
    
    return cost


### Cost-effective

In [7]:
def optimize(cost_function, init_params, steps):

    opt = qml.GradientDescentOptimizer(stepsize = 0.4) # Change this as you see fit

    params = init_params

    for i in range(steps):

        params = opt.step(cost_function, params)

    return params, cost_function(params)

initial_parameters = np.array([0.5, 0.5, 0.5, 0.5], requires_grad = True)
minimum = optimize(cost_function, initial_parameters, 100)[1]# An np.tensor of shape () containing the minimum of cost_function.
